# SwingSight Club-Head Detector Training

This notebook builds the club-head detector from the same master club-image dataset used by the five-way model and marking model. It exports `models/club_detector.pt`.


In [ ]:
import sys
import subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "ultralytics>=8.0.0", "pandas>=2.0.0", "pyyaml>=6.0.0", "pillow>=10.0.0",
    "--no-warn-script-location",
])
print("Training packages are ready.")


In [ ]:
from __future__ import annotations

from pathlib import Path
import shutil
import pandas as pd
import yaml

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Run this notebook inside the SwingSight-AI project.")

PROJECT_ROOT = find_project_root()
MASTER_DIR = PROJECT_ROOT / "data" / "club_training"
IMAGES_DIR = MASTER_DIR / "images"
MANIFEST_PATH = MASTER_DIR / "annotations" / "club_manifest.csv"
DERIVED_DIR = MASTER_DIR / "derived" / "club_detector"
MODEL_PATH = PROJECT_ROOT / "models" / "club_detector.pt"
EXPERIMENT_DIR = PROJECT_ROOT / "outputs" / "experiments" / "club_detector"

print("Master dataset:", MASTER_DIR)
print("Manifest:", MANIFEST_PATH)
print("App output:", MODEL_PATH)


## One shared club-image library

Both club models use the same master images. Keep the original images once, then derive task-specific train/validation folders from one annotation file.

```text
data/club_training/
  images/
    any_folder_you_want/
      image_0001.jpg
  annotations/
    club_manifest.csv
  derived/                    # created by these notebooks
```

The CSV uses one row per image. Required columns:

`image_path,split,five_way_label,head_x,head_y,head_w,head_h,marking_label,mark_x,mark_y,mark_w,mark_h`

- `image_path` is relative to `images/`.
- `split` is `train` or `val`. Keep a source image in one split only.
- `five_way_label` is `driver, wood, hybrid, iron,` or `wedge`.
- `head_*` is the normalized YOLO box for the club head.
- `marking_*` is the normalized box around only the number, letter, or loft. Leave every marking field blank when no readable marking is visible.

This is one dataset, not three copies of the same images.


In [ ]:
REQUIRED_COLUMNS = [
    "image_path", "split", "five_way_label",
    "head_x", "head_y", "head_w", "head_h",
    "marking_label", "mark_x", "mark_y", "mark_w", "mark_h",
]

if not MANIFEST_PATH.exists():
    MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(columns=REQUIRED_COLUMNS).to_csv(MANIFEST_PATH, index=False)
    raise FileNotFoundError(
        f"Created {MANIFEST_PATH}. Add your existing five-way images under {IMAGES_DIR} "
        "and annotate the club-head box for each image before rerunning."
    )

manifest = pd.read_csv(MANIFEST_PATH, keep_default_na=False)
missing = [column for column in REQUIRED_COLUMNS if column not in manifest.columns]
if missing:
    raise ValueError(f"Manifest is missing columns: {missing}")

manifest["split"] = manifest["split"].str.strip().str.lower()
if not manifest["split"].isin(["train", "val"]).all():
    raise ValueError("Every split must be train or val.")
for column in ["head_x", "head_y", "head_w", "head_h"]:
    manifest[column] = pd.to_numeric(manifest[column], errors="coerce")
if manifest[["head_x", "head_y", "head_w", "head_h"]].isna().any().any():
    raise ValueError("Every detector training image needs a complete normalized club-head box.")
if not ((manifest[["head_x", "head_y", "head_w", "head_h"]] >= 0).all().all()
        and (manifest[["head_x", "head_y", "head_w", "head_h"]] <= 1).all().all()):
    raise ValueError("YOLO box values must be normalized between 0 and 1.")

missing_images = [path for path in manifest["image_path"] if not (IMAGES_DIR / path).is_file()]
if missing_images:
    raise FileNotFoundError(f"Missing master images, first few: {missing_images[:10]}")

manifest.groupby(["split", "five_way_label"]).size().unstack(fill_value=0)


In [ ]:
# Build a derived YOLO folder from the master images and annotations.
# The originals remain in data/club_training/images.
for split in ("train", "val"):
    (DERIVED_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (DERIVED_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

for row in manifest.itertuples(index=False):
    source = IMAGES_DIR / row.image_path
    destination = DERIVED_DIR / "images" / row.split / source.name
    label_path = DERIVED_DIR / "labels" / row.split / f"{source.stem}.txt"
    shutil.copy2(source, destination)
    label_path.write_text(
        f"0 {row.head_x:.6f} {row.head_y:.6f} {row.head_w:.6f} {row.head_h:.6f}\n",
        encoding="utf-8",
    )

data_yaml = {
    "path": str(DERIVED_DIR),
    "train": "images/train",
    "val": "images/val",
    "names": {0: "club_head"},
}
yaml_path = DERIVED_DIR / "data.yaml"
yaml_path.write_text(yaml.safe_dump(data_yaml, sort_keys=False), encoding="utf-8")
print("Prepared detector images:", len(manifest))
print("YOLO config:", yaml_path)


## Train and export


In [ ]:
from ultralytics import YOLO

BASE_MODEL = "yolo11n.pt"
EPOCHS = 100
IMAGE_SIZE = 640

detector = YOLO(BASE_MODEL)
detector.train(
    data=str(yaml_path),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=-1,
    project=str(EXPERIMENT_DIR.parent),
    name=EXPERIMENT_DIR.name,
    exist_ok=True,
    patience=20,
    seed=42,
    pretrained=True,
)


In [ ]:
best_weights = EXPERIMENT_DIR / "weights" / "best.pt"
if not best_weights.exists():
    raise FileNotFoundError(f"Expected weights at {best_weights}")

MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(best_weights, MODEL_PATH)

trained_detector = YOLO(str(MODEL_PATH))
metrics = trained_detector.val(data=str(yaml_path), imgsz=IMAGE_SIZE)
print("Saved detector:", MODEL_PATH)
print(metrics)


Restart SwingSight after the export. The app will automatically use `models/club_detector.pt` to crop the club head before classification.
